# Ask 3 — Search by Words, and Where It Breaks

Score chunks by shared words, weight rare words most. Build it, watch it
win, then measure the failure class it can't touch: same meaning,
different words. Fully offline.

In [ ]:
# The pile: documents from the (fictional) Jefferson High School.
# Real enough to search, small enough to read whole.
PILE = {
 "handbook_academics": """S4.1 Grading scale. A 90-100, B 80-89, C 70-79, D 60-69.
Semester grades weight exams at 30 percent.
S4.2 Exam Retake Policy. This policy applies to final exams only. Students
receive one retake per semester, requested within ten school days. The
higher score stands.
S4.3 Grade appeals. Appeals go to the department head in writing within
fifteen school days of the posted grade.
S4.5 Late work. Assignments lose 10 percent per school day late, to a
maximum of 50 percent. Teachers may grant extensions for documented
emergencies.""",
 "handbook_schedule": """S2.0 Bell schedule. Regular days run eight periods,
8:15 AM to 3:20 PM.
S2.1 Wednesday schedule. Dismissal at 1:30 PM every Wednesday for staff
development.
S2.4 Late arrival. Students arriving after 8:30 AM sign in at the main
office with a note.""",
 "handbook_trips": """S5.1 Field trips require a signed permission form
submitted five school days in advance.
S5.2 Trip costs above 20 dollars qualify for the student activity fund.
S5.4 Chaperones must be approved district volunteers.""",
 "handbook_athletics": """S6.2 Eligibility. Athletes must hold a C average
during their season. Freshmen may try out for varsity teams.
S6.3 Petitions. A varsity roster spot for a freshman requires a coach's
petition to the athletic director.""",
 "robotics_minutes": """Robotics club meets Tuesdays in room 214. Regional
trip is April 18; bring your signed permission form by April 10. Dues are
15 dollars for the year.""",
 "clubs_list": """Active clubs: robotics (Tuesdays), debate (Thursdays),
art collective (Fridays), chess (lunch, library). Sign-up forms at the
student office.""",
 "bus_routes": """Routes 12 and 15 serve the north side. Final pickup at
4:45 PM outside door C. Activity buses run Tuesday and Thursday only.""",
 "cafeteria": """Lunch periods run 11:10, 11:55, and 12:40. Breakfast is
served from 7:40 AM. Menus post monthly on the food services page.""",
}
print(f"{len(PILE)} documents, {sum(len(t) for t in PILE.values())} characters total")

In [ ]:
def chunk_by_section(pile, overlap_sentences=1):
    """Cut on the S-section seams; carry a sentence of overlap across cuts."""
    chunks = []
    for doc, text in pile.items():
        parts, current, header = [], [], None
        for line in text.splitlines():
            if line.strip().startswith("S") and len(line) > 2 and line.strip()[1].isdigit():
                if current:
                    parts.append((header, " ".join(current)))
                header, current = line.strip().split()[0].rstrip("."), [line]
            else:
                current.append(line)
        if current:
            parts.append((header, " ".join(current)))
        for i, (header, body) in enumerate(parts):
            text_out = body
            if overlap_sentences and i > 0:
                prev_tail = parts[i-1][1].split(". ")[-1]
                text_out = prev_tail + " ... " + body
            chunks.append({"doc": doc, "section": header or doc, "text": " ".join(text_out.split())})
    return chunks

CHUNKS = chunk_by_section(PILE)
print(f"{len(CHUNKS)} chunks")
for c in CHUNKS[:3]:
    print(f"  [{c['doc']} {c['section']}] {c['text'][:70]}...")

## Rarity weighting, from scratch

Sharing "the" says nothing — every chunk has it. Sharing "retake" pins
the topic. Each shared word votes with weight 1/df: in how many chunks it
appears.

In [ ]:
import math, re, collections

def words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2]

# document frequency: in how many chunks does each word appear?
DF = collections.Counter()
for c in CHUNKS:
    for w in set(words(c["text"])):
        DF[w] += 1

def score(query, chunk):
    """Shared words, each weighted by rarity: rare words shout, common words whisper."""
    shared = set(words(query)) & set(words(chunk["text"]))
    return sum(1.0 / DF[w] for w in shared)

def retrieve(query, k=3):
    ranked = sorted(CHUNKS, key=lambda c: -score(query, c))
    return ranked[:k]

for c in retrieve("final exam retake"):
    print(f"{score('final exam retake', c):5.2f}  [{c['doc']} {c['section']}]  {c['text'][:60]}...")

## The scoreboard on a fair fight

The question and the answer share rare words. Word search earns its
decades of service:

In [ ]:
top = retrieve("How many final exam retakes do I get?")[0]
print("top hit:", top["doc"], top["section"])
assert top["section"] == "S4.2", "the retake question should retrieve S4.2"

## The silent failure

"Leave early midweek" IS "dismissal at 1:30 every Wednesday" — but they
share no meaningful words at all.
Watch the answer chunk lose:

In [ ]:
SYNONYM_Q = "when do we get to leave early midweek"
print(f"{'score':>6}  chunk")
for c in sorted(CHUNKS, key=lambda c: -score(SYNONYM_Q, c))[:5]:
    marker = "  <- THE ANSWER" if c["section"] == "S2.1" else ""
    print(f"{score(SYNONYM_Q, c):6.2f}  [{c['doc']} {c['section']}]{marker}")

ranked = sorted(CHUNKS, key=lambda c: -score(SYNONYM_Q, c))
answer_rank = [c["section"] for c in ranked].index("S2.1") + 1
print(f"\nThe answer ranked #{answer_rank}. Top-3 retrieval never sees it.")
print("No error. No warning. Just worse answers downstream - the failure is silent.")
assert answer_rank > 3, "the synonym question should miss under word search"

## Measure the failure rate honestly

In [ ]:
QUESTIONS = [
    ("How many final exam retakes do I get?", "S4.2"),
    ("What is the late work penalty?", "S4.5"),
    ("When are field trip permission forms due?", "S5.1"),
    ("When does robotics club meet?", "robotics_minutes"),
    ("What time is Wednesday dismissal?", "S2.1"),
    ("when do we get to leave early midweek", "S2.1"),      # synonym gap
    ("can we bail before the end of the day midweek", "S2.1"),  # worse gap
    ("Do ninth graders ever play varsity?", "S6.2"),        # "freshmen" != "ninth graders"
    ("How do I fight a bad grade?", "S4.3"),                # "appeal" != "fight"
    ("What does the ski trip cost?", None),                 # not in the pile
]

def hit(query, want, k=3):
    if want is None:
        return None
    got = retrieve(query, k)
    return any(c["section"] == want or c["doc"] == want for c in got)

results = [(q, hit(q, want)) for q, want in QUESTIONS]
hits = sum(1 for _, h in results if h)
print(f"word search: {hits} hits / {sum(1 for _, h in results if h is not None)} answerable questions\n")
for q, h in results:
    print(("  HIT   " if h else ("  n/a   " if h is None else "  MISS  ")), q)

## Try it

1. Every MISS above: was the answer absent, or present in different words?
   Sort them — that diagnosis is lesson 7's whole method.
2. Add a question of your own that you predict will miss, then check.
3. **Build turn-in:** your ten-question set over your pile with the
   hit/miss table and a diagnosis sentence per miss. Keep the set — it
   measures everything from here on.